In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.lines as mlines
import glob
import numpy as np
import xarray as xr
import rioxarray
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1 import AxesGrid
from matplotlib.colors import BoundaryNorm, ListedColormap
import cartopy.crs as ccrs
from pyproj import CRS, Transformer

## Plot fields

In [ ]:
def plot_anomaly_wr(wrname, save):
    
    #-------------------------------
    #layout
    
    fig = plt.figure(figsize = (20, 20))

    #-------------------------------
    #Set up a grid of 5x3 images. Each row has its colorbar.
    grid = AxesGrid(fig, 111,
                    nrows_ncols=(5, 3),
                    axes_pad=0.10,
                    share_all=True,
                    label_mode = "all",
                    cbar_location="right",
                    cbar_mode="edge",
                    cbar_size="5%",
                    cbar_pad="5%",
                    direction = "row"
                   )

    #load study region
    study_area = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_32E.shp")
    regions = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/Region_Ten.shp")

    #load land mask
    land_mask_binary = xr.open_dataarray("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Land_Mask_Binary_CERRA_reproject_EPSG4326_study_area_32E.nc")


    #-------------------------------
    #colorlist

    #BuRd 10 (for mx2t, 10si, fwi)
    colorlist1 = ["#053061", "#2166ac", "#4393c3", "#92c5de", "#d1e5f0", "#fddbc7", "#f4a582", "#d6604d", "#b2182b", "#67001f"]
    cmap1 = ListedColormap(colorlist1)
    bounds1 = [-0.75, -0.6, -0.45, -0.3, -0.15, 0, 0.15, 0.3, 0.45, 0.6, 0.75] #len(bounds) = len(colors) + 1
    norm1 = BoundaryNorm(bounds1, cmap1.N)

    #RdBu 10 (for tp, 2r)
    colorlist2 = colorlist1[::-1]
    cmap2 = ListedColormap(colorlist2)
    bounds2 = [-0.75, -0.6, -0.45, -0.3, -0.15, 0, 0.15, 0.3, 0.45, 0.6, 0.75] #len(bounds) = len(colors) + 1
    norm2 = BoundaryNorm(bounds2, cmap2.N)

    #label
    label_dict = dict(zip(["10si", "tp", "2r", "mx2t", "fwi"], ["Wind Speed", "Total Precipitation", "Relative Humidity", "Max Temperature", "Fire Weather Index"]))
    sub_lb_dict = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)", "(h)", "(i)", "(j)", "(k)", "(l)", "(m)", "(n)", "(o)"]

    
    #-------------------------------
    i = 0
    
    for varname in ["mx2t", "2r", "10si", "tp", "fwi"]:
        
        #-------------------------------
        # specify colormap
        if varname in ["mx2t", "10si", "fwi"]:
            cmap = cmap1
            norm = norm1
            bounds = bounds1
        else:
            cmap = cmap2
            norm = norm2
            bounds = bounds2
            
        
        for season in ["MAM", "JJA", "SON"]:

            #-------------------------------
            #load and reproject anomaly data
            indir = "/net/rain/hyclimm/data/projects/SynFire/WP1/CERRA_Variable_Anomaly_by_Weather_Regime/" 
            var = xr.open_dataarray(os.path.join(indir, f"Anomaly_{varname}_{season}_{wrname}.nc"))

            var = var.rio.write_crs("+proj=lcc +lat_1=50 +lat_2=50 +lon_0=8 +lat_0=50 +x_0=2937000.506058639 +y_0=2937000.470434457 +datum=WGS84")
            var_lonlat = var.rio.reproject("EPSG:4326")
            var_lonlat = var_lonlat.sel(x = slice(-12, 34), y = slice(72, 33))
        
            #mask out ocean
            var_lonlat = var_lonlat.where(land_mask_binary)
        
            
            #spatial extent
            lon = var_lonlat.x.values
            lat = var_lonlat.y.values
            extent = [lon.min(), lon.max(), lat.min(), lat.max()]
                
                
            #-------------------------------
            im = grid[i].imshow(var_lonlat, extent=extent, norm = norm, cmap = cmap, origin = "upper")
            study_area.boundary.plot(ax = grid[i], edgecolor = "black", linewidth = 0.2)
            regions.boundary.plot(ax = grid[i], edgecolor = "black", linewidth = 0.5)

            #turn off x, y label
            grid[i].tick_params(left = False, labelleft = False, bottom = False, labelbottom = False)

            #add colorbar
            if i%3 == 2:
                grid.cbar_axes[i//3].colorbar(im, extend='both', boundaries=bounds, ticks=bounds).ax.tick_params(labelsize=10)

            #add y label
            if not i%3:
                grid[i].set_ylabel(label_dict[varname], fontsize = 16)

            #add season label
            if varname == "mx2t":
                grid[i].set_title("Spring" if i == 0 else "Summer" if i == 1 else "Fall", fontsize = 16)

            #add subpanel label
            grid[i].text(0.03, 0.92, sub_lb_dict[i], transform = grid[i].transAxes, fontsize = 16)
            
            i += 1

    
    for cax in grid.cbar_axes:
        cax.axis[cax.orientation].set_label('Standardized Anomaly (-)')
        cax.set_yticks([-0.75, -0.6, -0.45, -0.3, -0.15, 0, 0.15, 0.3, 0.45, 0.6, 0.75])
        cax.set_yticklabels(["-0.75", "-0.60", "-0.45", "-0.30", "-0.15", "0", "0.15", "0.30", "0.45", "0.60", "0.75"])
        cax.axis[cax.orientation].label.set_size(13)
    
    if save:
        plt.savefig(f"/home/lixinh/SynFire_Scripts/WP1/Figures/{wrname}_atmos_standardized_anomaly.png", dpi = 600, bbox_inches = "tight")

    plt.show()

In [ ]:
for wr in ["AT", "ZO", "ScTr", "AR", "EuBL", "ScBL", "GL", "no"]:
    plot_anomaly_wr(wr, True)

# Plot averages (latitude weighted)

### get data

In [ ]:
#load region masks
region_mask = xr.open_dataarray("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Region_Mask_CERRA_reproject_EPSG4326_Ten.nc")
region_mask = region_mask.rio.write_crs('EPSG:4326')
lcc_crs_proj = ccrs.LambertConformal(
    central_longitude=8, central_latitude=50, 
    standard_parallels=(50, 50),  
    false_easting=2937000.506058639, 
    false_northing=2937000.470434457
)
region_mask = region_mask.rio.reproject(lcc_crs_proj)
region_mask_key = dict(zip(["BI", "IP", "FR", "ME", "AL", "SEA", "NEA", "SC", "WMD", "EMD"], [i for i in range(10)]))

#expand region masks to the CERRA domain
var = xr.open_dataset(f"/net/rain/hyclimm/data/projects/fofix/standardize_Cerra_empirically_original_res_from_shell/2t_cerra_grid_2001-01-01_2020-12-31.nc")
var = var.rio.write_crs(lcc_crs_proj)
region_mask = region_mask.interp(x = var.x, y = var.y, method = "nearest")

In [ ]:
rows = []

for wrname in ["AT", "ZO", "ScTr", "AR", "EuBL", "ScBL", "GL", "no"]:

    for season in ['MAM', 'JJA', 'SON']:

        for reg in ["BI", "IP", "FR", "ME", "AL", "SEA", "NEA", "SC", "WMD", "EMD"]:
            
            val = dict()
            
            for varname in ['mx2t', '2r', '10si']:
                
                var = xr.open_dataarray(f'/net/rain/hyclimm/data/projects/SynFire/WP1/CERRA_Variable_Anomaly_by_Weather_Regime/Anomaly_{varname}_{season}_{wrname}.nc')
                var = var.rio.write_crs(lcc_crs_proj)

                #add lon, lat grids
                lcc_crs = {'grid_mapping_name': 'lambert_conformal_conic',
                           'standard_parallel': 50.0,
                           'longitude_of_central_meridian': 8.0,
                           'latitude_of_projection_origin': 50.0,
                           'earth_radius': 6371229.0,
                           'false_easting': 2937000.506058639,
                           'false_northing': 2937000.470434457,
                           'longitudeOfFirstGridPointInDegrees': 342.514057,
                           'latitudeOfFirstGridPointInDegrees': 20.292281,
                          }
                
                crs_proj = CRS.from_cf(lcc_crs)
                crs_geo = CRS.from_epsg(4326)
                transformer = Transformer.from_crs(crs_from = crs_proj, crs_to = crs_geo, always_xy=True)
                
                x = var['x'].values
                y = var['y'].values
                
                xx, yy = np.meshgrid(x, y)  #inputs of length M and N, the outputs are of shape (N, M) by default
                
                lon, lat = transformer.transform(xx, yy)
                
                var = var.assign_coords(
                lon = (('y', 'x'), lon),
                lat = (('y', 'x'), lat)
                )

                #calculate weight
                weights = np.cos(np.deg2rad(var.lat))
                weights.name = 'weights'
                
                var_reg = var.where(region_mask == region_mask_key[reg], drop = True)


                
                #weighted mean
                var_reg_weighted_mean = var_reg.weighted(weights).mean().values
                
                #weighted standard deviation
                var_reg_weighted_variance = (
                    (var_reg - var_reg_weighted_mean)**2
                ).weighted(weights).mean()
                
                var_reg_weighted_std = np.sqrt(var_reg_weighted_variance).values

                

                val.update({f'{varname}_mean': var_reg_weighted_mean, f'{varname}_std': var_reg_weighted_std})


            val.update({'region': reg, 'season': season, 'wr': wrname})
            rows.append(val)

df = pd.DataFrame(rows)

In [ ]:
#save
df.to_csv('/net/rain/hyclimm/data/projects/SynFire/WP1/CERRA_Variable_Anomaly_by_Weather_Regime/Standardized_anomaly_mx2t_2r_10si_mean_std_by_region_and_weather_regime_lat_weighted.csv', index = False)

In [ ]:
df = pd.read_csv('/net/rain/hyclimm/data/projects/SynFire/WP1/CERRA_Variable_Anomaly_by_Weather_Regime/Standardized_anomaly_mx2t_2r_10si_mean_std_by_region_and_weather_regime_lat_weighted.csv')

fig, axes = plt.subplots(3, 3, figsize = (14, 14))
axes = axes.flatten()
axes[-1].axis('off')

#marker for each region
marker_dict = {'IP':'o',
               'EMD':'v',
               'AL':'>',
               'BI':'x',
               'NEA':'d',
               'WMD':'^',
               'FR':'<',
               'SEA': 'P',
               'ME':'D',
               'SC':'s',
              }

#subplot title
title_dict = {'EuBL': '(a) EuBL',
              'ScBL': '(b) ScBL',
              'ZO': '(c) ZO',
              'AT': '(d) AT',
              'ScTr': '(e) ScTr',
              'AR': '(f) AR',
              'GL': '(g) GL',
              'no': '(h) no'}



#highlight
mam_dict = {'EuBL': ['BI', 'SC', 'NEA', 'ME', 'SEA', 'IP'],
            'ScBL': ['BI', 'SC', 'NEA'],
            'ZO': ['IP', 'NEA', 'SEA', 'EMD'],
            'ScTr': ['IP', 'EMD']}

jja_dict = {'ScBL': ['SC', 'WMD', 'EMD', 'SEA', 'NEA'],
            'AR': ['FR', 'IP', 'WMD', 'EMD', 'NEA', 'SC'],
            'AT': ['SEA', 'WMD', 'EMD'],
            'EuBL': ['SC', 'NEA'],
            'ZO': ['SC', 'NEA'],
            'no': ['IP', 'WMD', 'EMD']}

son_dict = {'EuBL': ['IP', 'NEA'],
            'ZO': ['SEA', 'EMD', 'NEA'],
            'ScTr': ['IP', 'WMD', 'EMD']}

#season color
color_dict = {'MAM': '#0d98ba',
              'JJA': '#fe7d6a',  #light pink
              'SON': '#895129'}
 
for wrname, ax in zip(['EuBL', 'ScBL', 'ZO', 'AT',  'ScTr', 'AR', 'GL', 'no'], axes[:-1]):
    
    ax.axvline(x = 0, linestyle = '--', color = '#d3d3d3', zorder = 0)
    ax.axhline(y = 0, linestyle = '--', color = '#d3d3d3', zorder = 0)
    ax.set_aspect('equal')
    
    for season in ['MAM', 'JJA', 'SON']:

        dfsub = df[(df['wr'] == wrname) & (df['season'] == season)]
        season_dict = mam_dict if season == 'MAM' else jja_dict if season == 'JJA' else son_dict

        

        for _, row in dfsub.iterrows():
            reg = row['region']

            #set colors
            if wrname in season_dict.keys() and reg in season_dict[wrname]:
                col = color_dict[season]
                zorder = 2
            else:
                col = '#e0e0e0'
                zorder = 1
                
            
            x = row['mx2t_mean']
            xerr = row['mx2t_std']
            y = row['2r_mean']
            yerr = row['2r_std']

            ws = row['10si_mean']
            ls = '-' if ws > 0 else '--'

            ax.plot(x, y, marker = marker_dict[reg], markersize = 8, color = col, zorder = zorder)
            ax.plot([x-xerr, x+xerr], [y, y], linestyle = ls, color = col, zorder = zorder)
            ax.plot([x, x], [y-yerr, y+yerr], linestyle = ls, color = col, zorder = zorder)

    ax.set_xlim(-1, 1)
    ax.set_xticks([-1.0, -0.5, 0, 0.5, 1.0])
    ax.set_xticklabels(["-1.0", "-0.5", "0", "0.5", "1.0"])
    ax.set_ylim(-1, 1)
    ax.set_yticks([-1.0, -0.5, 0, 0.5, 1.0])
    ax.set_yticklabels(["-1.0", "-0.5", "0", "0.5", "1.0"])
    ax.tick_params(axis = "both", labelsize = 12)
    ax.set_xlabel('Maximum Temperature (-)', fontsize = 15, labelpad = 0)
    ax.set_ylabel('Mean Relative Humidity (-)', fontsize = 15, labelpad = 0)
    ax.text(0.05, 0.9, title_dict[wrname], transform = ax.transAxes, fontsize = 15)

#wind speed legend
x1, y1 = -0.85, -0.1
x2, y2 = 0.15, -0.1

axes[-1].set_xlim(-1, 1)
axes[-1].set_ylim(-1, 1)
axes[-1].plot([x1 - 0.15, x1 + 0.15], [y1, y1], color='black', linestyle='-')  
axes[-1].plot([x1, x1], [y1 - 0.15, y1 + 0.15], color='black', linestyle='-')  
axes[-1].text(-0.7, -0.1, 'Positive', fontsize = 15, va = 'center')

axes[-1].plot([x2 - 0.15,  x2 + 0.15], [y2, y2], color='black', linestyle='--')  
axes[-1].plot([x2, x2], [y2 - 0.15, y2 + 0.15], color='black', linestyle='--')  
axes[-1].text(0.3, -0.1, 'Negative', fontsize = 15, va = 'center')

axes[-1].text(0, 0.1, 'Wind Speed', fontsize = 18, ha = 'center')

#other legends
hd_reg = [mlines.Line2D([0], [0], marker=marker_dict[region], color='black',
                       label= 'NEE' if region == 'NEA' else 'SEE' if region == 'SEA' else 'CE' if region == 'ME' else region, linestyle='None', markersize=8) for region in marker_dict.keys()]

hd_season = [mlines.Line2D([0], [0], marker='s', color=cl,
                       label=lb, linestyle='None', markersize=8) for (cl, lb) in [(color_dict['MAM'], 'Spring'), (color_dict['JJA'], 'Summer'), (color_dict['SON'], 'Fall'), ('#e0e0e0', 'Not Shown')]]

lg_reg = axes[-1].legend(handles = hd_reg, loc = 'lower center', frameon = False, ncol = 5, bbox_to_anchor = (0.5, 0), handletextpad = -0.5, 
           fontsize = 13, columnspacing = 0.5, title = 'Region', title_fontsize = 18)

lg_season = axes[-1].legend(handles=hd_season, title='Season', loc='lower center', bbox_to_anchor = (0.45, 0.65), ncol = 2, frameon = False, fontsize = 15, title_fontsize = 18,
           handletextpad = -0.5, columnspacing = 1.5)

axes[-1].add_artist(lg_reg)

plt.subplots_adjust(wspace = 0.28)

plt.savefig('/home/lixinh/SynFire_Scripts/WP1/Figures/v4_5_weather_regime_seasonal_anomaly.png', dpi = 600, bbox_inches = 'tight')